In [1]:

import libzim
from zimscraperlib import zim
# Open the ZIM file
reader = zim.Archive("/home/scott/Downloads/gutenberg/gutenberg_en_all_2025-11.zim")

# First, let's explore what's in it
print(f"Entry count: {reader.entry_count}")
print(f"Main entry path: {reader.main_entry.path}")

# --- Option 1: Browse entries by index ---
# Grab an arbitrary entry (e.g., the 500th)
entry = reader.get_random_entry()
#print(entry.path, entry.title)

# Read its content (returns bytes)
content = bytes(entry.get_item().content)
#print(content[:5000].decode("utf-8", errors="replace"))

# --- Option 2: Access a known path directly ---
# ZIM files from Kiwix/Gutenberg typically use paths like:
#   A/12345.html  (article namespace)
entry = reader.get_entry_by_path('Jupiter found.72408')
content = bytes(entry.get_item().content).decode("utf-8", errors="replace")

# --- Option 3: Full-text search (if the ZIM has a search index) ---
if reader.has_fulltext_index:
    searcher = libzim.Searcher(reader)
    query = libzim.Query().set_query("Moby Dick")
    search = searcher.search(query)
    results = list(search.getResults(0, 10))
    for path in results:
        e = reader.get_entry_by_path(path)
        print(e.path, e.title)
        print("-")

# --- Option 4: Iterate all entries and filter by title ---
for i in range(min(reader.entry_count, 10000)):
    e = reader.get_entry_by_id(i)
    if "Moby" in e.title:
        content = bytes(e.get_item().content).decode("utf-8", errors="replace")
        print(content[:1000])
        break

Entry count: 1562228
Main entry path: mainPage
Moby-Dick; or, The Whale.15 Moby-Dick; or, The Whale
-
Moby Dick; Or, The Whale.2701 Moby Dick; Or, The Whale
-
Moby Dick; Or, The Whale.2489 Moby Dick; Or, The Whale
-
Herman Melville, Mariner and Mystic.50461 Herman Melville, Mariner and Mystic
-
The Merman and the Figure-Head.53901 The Merman and the Figure-Head
-
Eva's Adventures in Shadow-Land.53899 Eva's Adventures in Shadow-Land
-
Studies in Classic American Literature.60547 Studies in Classic American Literature
-
Great Sea Stories.18405 Great Sea Stories
-
Aspects of the novel.70492 Aspects of the novel
-
Excursions in Victorian Bibliography.53118 Excursions in Victorian Bibliography
-


In [2]:
from collections import Counter
counts = Counter()
targets = []
for i in range(min(reader.entry_count, 10000)):
    e = reader.get_entry_by_id(i)
    suffix = e.path.split(".")[-1]
    if suffix.isdigit():
        continue
    counts[suffix] += 1
counts

Counter({'jpg': 4993,
         'png': 3369,
         'gif': 792,
         'epub': 269,
         'JPG': 11,
         'jpeg': 10,
         'css': 6,
         'pdf': 3})

In [3]:
targets = []
for i in range(min(reader.entry_count, 10000)):
    e = reader.get_entry_by_id(i)
    suffix = e.path.split(".")[-1]
    if suffix.isdigit():
        if "_cover" not in e.path:
            targets.append(e.path)
print(len(targets))
print(targets[:5])

278
['!Tention: A Story of Boy-Life during the Peninsular War.21374', '"\'Tis Sixty Years Since".9996', '"1683-1920".50075', '"1812" : $b Napoleon I in Russia.51418', '"1914".66846']


In [4]:
entry = reader.get_entry_by_path(targets[0])
entry

Entry(url=!Tention: A Story of Boy-Life during the Peninsular War.21374, title=!Tention: A Story of Boy-Life during the Peninsular War)

In [5]:
bytes(entry.get_item().content[:10000])

b'<html lang="en">\n<head><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>\n<meta/><style>\n#pg-header div, #pg-footer div {\n    all: initial;\n    display: block;\n    margin-top: 1em;\n    margin-bottom: 1em;\n    margin-left: 2em;\n}\n#pg-footer div.agate {\n    font-size: 90%;\n    margin-top: 0;\n    margin-bottom: 0;\n    text-align: center;\n}\n#pg-footer li {\n    all: initial;\n    display: block;\n    margin-top: 1em;\n    margin-bottom: 1em;\n    text-indent: -0.6em;\n}\n#pg-footer div.secthead {\n    font-size: 110%;\n    font-weight: bold;\n}\n#pg-footer #project-gutenberg-license {\n    font-size: 110%;\n    margin-top: 0;\n    margin-bottom: 0;\n    text-align: center;\n}\n#pg-header-heading {\n    all: inherit;\n    text-align: center;\n    font-size: 120%;\n    font-weight:bold;\n}\n#pg-footer-heading {\n    all: inherit;\n    text-align: center;\n    font-size: 120%;\n    font-weight: normal;\n    margin-top: 0;\n    margin-bottom: 0;\n}\n#pg-head

In [6]:
from bs4 import BeautifulSoup

In [7]:
soup = BeautifulSoup(bytes(entry.get_item().content))
pages = soup.find_all("div", class_="bodytext")
len(pages)

45

In [8]:
pages[0].text

'\n\nChapter One.\nTo save a Comrade.\nA sharp volley, which ran echoing along the ravine, then another, just as the faint bluish smoke from some hundred or two muskets floated up into the bright sunshine from amidst the scattered chestnuts and cork-trees that filled the lower part of the beautiful gorge, where, now hidden, now flashing out and scattering the rays of the sun, a torrent roared and foamed along its rocky course onward towards its junction with the great Spanish river whose destination was the sea.\nAgain another ragged volley; and this was followed by a few dull, heavy-sounding single shots, which came evidently from a skirmishing party which was working its way along the steep slope across the river.\nThere was no responsive platoon reply to the volley, but the skirmishing shots were answered directly by crack! crack! crack! the reports that sounded strangely different to those heavy, dull musket-shots which came from near at hand, and hardly needed glimpses of dark-gre

In [9]:
pages[1].text

'\n\nChapter Two.\nPoor Punch.\nPrivate Gray, of his Majesty’s —th Rifles,—wrenched himself round once more, pressed aside a clump of heathery growth, crawled quickly about a couple of yards, and found himself lying face to face with the bugler of his company.\n“Why, Punch, lad!” he said, “not hurt much, are you?”\n“That you, Private Gray?”\n“Yes. But tell me, are you wounded?”\n“Yes!” half-groaned the boy; and then with a sudden access of excitement, “Here, I say, where’s my bugle?”\n“Oh, never mind your bugle. Where are you hurt?” cried the boy’s comrade.\n“In my bugle—I mean, somewhere in my back. But where’s my instrument?”\n“There it is, in the grass, hanging by the cord.”\n“Oh, that’s better,” groaned the boy. “I thought all our chaps had gone on and left me to die.”\n“And now you see that they hav’n’t,” said the boy’s companion. “There, don’t try to move. We mustn’t be seen.”\n“Yes,” almost babbled the boy, speaking piteously, “I thought they had all gone, and left me here. I di

In [10]:
import chromadb
client = chromadb.Client()

In [11]:
from chromadb.utils.embedding_functions.ollama_embedding_function import (
    OllamaEmbeddingFunction,
)

ollama_ef = OllamaEmbeddingFunction(
    url="http://localhost:11434",
    model_name="deepseek-r1:1.5b",
)

embeddings = ollama_ef(["This is my first text to embed",
                        "This is my second document"])

In [12]:
collection = client.create_collection(
    name="collection",
    embedding_function=ollama_ef,
    configuration={"hnsw": {"space": "cosine"}}
)

In [13]:
def entry_text_iter(entry):
    soup = BeautifulSoup(bytes(entry.get_item().content))
    pages = soup.find_all("div", class_="bodytext")
    for page in pages:
        yield page.text

In [15]:
for i in range(min(reader.entry_count, 1000)):
    e = reader.get_entry_by_id(i)
    suffix = e.path.split(".")[-1]
    if suffix.isdigit():
        if "_cover" not in e.path:
            print(i, e.title)
            ids = []
            texts = []
            for i, text in enumerate(entry_text_iter(e)):
                ids.append(f"{e.title}_{i}")
                texts.append(text)
            if texts:
                collection.add(ids=ids, documents=texts)

0 !Tention: A Story of Boy-Life during the Peninsular War
3 "'Tis Sixty Years Since"
6 "1683-1920"
9 "1812" : $b Napoleon I in Russia
12 "1914"
15 "95% perfect" : $b The older residences at Nantucket
18 "A Cathcart or a Riggs?"
21 "A Modern Hercules," the Tale of a Sculptress
24 "A Most Unholy Trade," Being Letters on the Drama by Henry James
27 "A Soldier Of The Empire"
30 "Abe" Lincoln's Anecdotes and Stories
33 "About My Father's Business": Work Amidst the Sick, the Sad, and the Sorrowing
36 "All's Well"; or, Alice's Victory
39 "All's not Gold that Glitters;" or, The Young Californian
42 "America for Americans!"
45 "And That's How It Was, Officer"
48 "And they thought we wouldn't fight"
51 "Around the Circle": One Thousand Miles Through the Rocky Mountains
54 "As Gold in the Furnace" : A College Story
57 "Ask Mamma"; or, The Richest Commoner In England
60 "Back from hell"
63 "Barbarous Soviet Russia"
66 "Bear ye one another's burdens." A Plain Sermon on the Lancashire Distress
69 "B

In [25]:
bytes(entry.get_item().content[:3000])

b'<html lang="en">\n<head><meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>\n<meta/><style>\n#pg-header div, #pg-footer div {\n    all: initial;\n    display: block;\n    margin-top: 1em;\n    margin-bottom: 1em;\n    margin-left: 2em;\n}\n#pg-footer div.agate {\n    font-size: 90%;\n    margin-top: 0;\n    margin-bottom: 0;\n    text-align: center;\n}\n#pg-footer li {\n    all: initial;\n    display: block;\n    margin-top: 1em;\n    margin-bottom: 1em;\n    text-indent: -0.6em;\n}\n#pg-footer div.secthead {\n    font-size: 110%;\n    font-weight: bold;\n}\n#pg-footer #project-gutenberg-license {\n    font-size: 110%;\n    margin-top: 0;\n    margin-bottom: 0;\n    text-align: center;\n}\n#pg-header-heading {\n    all: inherit;\n    text-align: center;\n    font-size: 120%;\n    font-weight:bold;\n}\n#pg-footer-heading {\n    all: inherit;\n    text-align: center;\n    font-size: 120%;\n    font-weight: normal;\n    margin-top: 0;\n    margin-bottom: 0;\n}\n#pg-head

In [18]:
collection.count()

127

In [21]:
collection.query(
    query_texts=["A sharp volley, which ran echoing along the ravine, then another, just as the faint bluish smoke from some hundred or two muskets floated up into the bright sunshine from amidst the scattered chestnuts and cork-trees that filled the lower part of the beautiful gorge, where, now hidden, now flashing out and scattering the rays of the sun, a torrent roared and foamed along its rocky course onward towards its junction with the great Spanish river whose destination was the sea.\nAgain another ragged volley; and this was followed by a few dull, heavy-sounding single shots"]
)

{'ids': [['!Tention: A Story of Boy-Life during the Peninsular War_1',
   '!Tention: A Story of Boy-Life during the Peninsular War_7',
   "'Tween Snow and Fire: A Tale of the Last Kafir War_44",
   "'Tween Snow and Fire: A Tale of the Last Kafir War_12",
   "'Tween Snow and Fire: A Tale of the Last Kafir War_31",
   "'Tween Snow and Fire: A Tale of the Last Kafir War_33",
   '!Tention: A Story of Boy-Life during the Peninsular War_22',
   '!Tention: A Story of Boy-Life during the Peninsular War_5',
   '!Tention: A Story of Boy-Life during the Peninsular War_34',
   "'Tween Snow and Fire: A Tale of the Last Kafir War_45"]],
 'embeddings': None,
 'documents': [['\n\nChapter Two.\nPoor Punch.\nPrivate Gray, of his Majesty’s —th Rifles,—wrenched himself round once more, pressed aside a clump of heathery growth, crawled quickly about a couple of yards, and found himself lying face to face with the bugler of his company.\n“Why, Punch, lad!” he said, “not hurt much, are you?”\n“That you, Priv

In [22]:
collection.get(
   where_document={"$contains": "Chapter One"}
)

{'ids': ['!Tention: A Story of Boy-Life during the Peninsular War_0',
  '"All\'s Well"; or, Alice\'s Victory_0',
  "'Tween Snow and Fire: A Tale of the Last Kafir War_0"],
 'embeddings': None,
 'documents': ['\n\nChapter One.\nTo save a Comrade.\nA sharp volley, which ran echoing along the ravine, then another, just as the faint bluish smoke from some hundred or two muskets floated up into the bright sunshine from amidst the scattered chestnuts and cork-trees that filled the lower part of the beautiful gorge, where, now hidden, now flashing out and scattering the rays of the sun, a torrent roared and foamed along its rocky course onward towards its junction with the great Spanish river whose destination was the sea.\nAgain another ragged volley; and this was followed by a few dull, heavy-sounding single shots, which came evidently from a skirmishing party which was working its way along the steep slope across the river.\nThere was no responsive platoon reply to the volley, but the skir

In [26]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.tools import tool

# Load AI Model
llm = ChatOllama(model="ministral-3:latest")

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='21a0d269-1b66-4f6b-a11e-cde5859177aa'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'ministral-3:latest', 'created_at': '2026-03-28T17:59:18.480551552Z', 'done': True, 'done_reason': 'stop', 'total_duration': 36185427320, 'load_duration': 34182982575, 'prompt_eval_count': 65, 'prompt_eval_duration': 343459069, 'eval_count': 13, 'eval_duration': 1555522571, 'logprobs': None, 'model_name': 'ministral-3:latest', 'model_provider': 'ollama'}, id='lc_run--019d3599-2576-7941-bb19-c35bfaeca2ae-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'fa6bd4be-2f2f-416a-b5cd-52f102c67af7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 13, 'total_tokens': 78}),
  ToolMessage(content="It's always sunny in San Francisco!", name='get_weather', id='e4e24abf-2834-493b-a42a-7c6586a6447a

AIMessage(content='And\nWorld\nWas\nChanged\nchapter\nthirteen', additional_kwargs={}, response_metadata={'model': 'ministral-3:latest', 'created_at': '2026-03-28T18:49:35.031395144Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2990266604, 'load_duration': 241969976, 'prompt_eval_count': 591, 'prompt_eval_duration': 1190161714, 'eval_count': 14, 'eval_duration': 1549297719, 'logprobs': None, 'model_name': 'ministral-3:latest', 'model_provider': 'ollama'}, id='lc_run--019d35c7-ae83-7663-bc58-c0856cb12e12-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 591, 'output_tokens': 14, 'total_tokens': 605})

In [58]:
extract_keywords("What happens in chapter 13 of And The WOrld Was Changed?")

'And, The World Was Changed, chapter 13'

In [76]:
PROMPT_TEMPLATE = """
You will be presented with a question. Answer it using only the following context:
{context}
 - -
Answer this question based on the above context: {query}
"""

def query_db(query: str) -> list[str]:
    results = collection.query(query_texts=query, n_results=3)
    returned = []
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(dist)
        if dist < 0.5:
            returned.append(doc)
    return returned

def extract_keywords(query: str) -> list[str]:
    result = llm.invoke(
        f"What keywords from this question would you use to query a database for more information? Respond only with a comma-delimited list of words or phrases from the question itself: '{query}'"
    )
    terms = []
    for t in result.content.split(","):
        t = "".join(c for c in t if c.isalnum() or c == " ")
        terms.append(t)
    return terms
    
def rag(query: str):
    keywords = extract_keywords(query)
    print(keywords)
    keyword_search = [{"$contains": kw} for kw in keywords]
    print(keyword_search)
    documents = collection.get(where_document={
        "$or": keyword_search
    })["documents"]
    #results = query_db(query)
    #if not results:
    #    return None
    context = "\n---\n".join(documents)
    formatted = PROMPT_TEMPLATE.format(context=context, query=query)
    print(formatted)
    result = llm.invoke(formatted)
    return result
rag("What happens in Chapter Thirteen of ...And The World Was Changed?")

['Chapter Thirteen', 'And The World Was Changed']
[{'$contains': 'Chapter Thirteen'}, {'$contains': 'And The World Was Changed'}]

You will be presented with a question. Answer it using only the following context:


Chapter Thirteen.
“Look out, Comrade!”
“Hooray!” cried Punch, wrenching his head round and stretching one hand towards their visitor, who stepped in, put the basket she carried upon the bed, and placed her hand upon her side, breathing hard as if she were in pain.
“Why, you have been running,” cried Punch, looking at her reproachfully. “It was all right on you, and you are a good little lass to come, but you shouldn’t have run so fast. ’Tain’t good.”
As the girl began to recover her breath she showed her white teeth and nodded merrily at the wounded boy; and then, as if she had grasped his meaning, she turned to Pen, caught up the basket, and began rapidly to take out its contents, which consisted first of bunches of grapes, a few oranges, and from beneath them a piece of t

AIMessage(content='In the provided excerpt, we are not given the full content of **Chapter Thirteen** of *And the World Was Changed* by **Gordon Cuppleditch**, but we can infer some key events and developments based on the context given in the passage.\n\nFrom the context, here’s what likely happens in **Chapter Thirteen**:\n\n1. **Preparation for the Campaign**:\n   The chapter likely continues the tension and preparations for the impending conflict between the British settlers and the Kafirs. The conversation between Eustace Milne, Tom Carhayes, and Mr. Hoste suggests that the men are discussing their roles in the coming war, including the movement of livestock and the involvement of Nteya, a Kafir servant.\n\n2. **Eustace and Eanswyth’s Separation**:\n   The chapter probably focuses on the emotional tension between Eustace and Eanswyth. Their relationship is in a delicate state, with Eanswyth feeling overwhelmed by her newfound love for Eustace and the impending separation due to th

In [71]:
"foo".isalnum()

True

In [40]:
query_db("Chapter Thirty Four.\nFrom Death and—to Death.\nShe realised it at length—realised that this was no visitant from the spirit-world conjured up in answer to her impassioned prayer, but her lover himself, alive and unharmed.")

0.49641603231430054
0.6399489641189575
0.6615346670150757


['\n\nChapter Twenty Three.\nThe Use of Latin.\n“There! Ahoy!” shouted Punch, and the black figure slowly raised his head and began to look round till he was gazing in quite the opposite direction to where the boy was hurrying towards him, and Punch had a full view of the stranger’s back and a ruddy-brown roll of fat flesh which seemed to be supporting a curious old hat, looking like a rusty old stove-pipe, perched horizontally upon the wearer’s head.\n“Hi! Not that way! Look this!” cried Punch as he closed up. “Here, I say, where’s the nearest village?”\nThe stove-pipe turned slowly round, and Punch found himself face to face with a plump-looking little man who slowly closed the book he carried and tucked it inside his shabby gown.\n“Morning!” said Punch.\nThe little man bowed slowly and with some show of dignity, and then gazed sternly in the boy’s face and waited.\n“I said good-morning, sir,” said the boy; and then to himself, “what a rum-looking little chap!—Can you tell me—”\nPunc

In [30]:
results

{'ids': [["'Tween Snow and Fire: A Tale of the Last Kafir War_33",
   "'Tween Snow and Fire: A Tale of the Last Kafir War_12",
   '!Tention: A Story of Boy-Life during the Peninsular War_1']],
 'embeddings': None,
 'documents': [['\n\nChapter Thirty Four.\nFrom Death and—to Death.\nShe realised it at length—realised that this was no visitant from the spirit-world conjured up in answer to her impassioned prayer, but her lover himself, alive and unharmed. She had thrown herself upon his breast, and clung to him with all her strength, sobbing passionately—clung to him as if even then afraid that he might vanish as suddenly as he had appeared.\n“My love, my love,” he murmured in that low magnetic tone which she knew so well, and which thrilled her to the heart’s core. “Calm those poor nerves, my darling, and rest on the sweetness of our meeting. We met—our hearts met first on this very spot. Now they meet once more, never again to part.”\nStill her feeling was too strong for words; she cou